# Economy Sim

A notebook which launches an economy simulator.

In [1]:
import uuid
from decimal import Decimal
import datetime
from pathlib import Path

from configs.config import GLOBAL_ECONOMY_LOG_DIR
from utils.logger import create_custom_logger, override_print_with_logger

from exceptions.not_enough_money_error import NotEnoughMoneyError

from models.enums import CompetitionModel
from models.config_model import WholesalerConfig, RetailerConfig, ConsumerConfig


### Logging

This cell creates a logger, and attaches it to `print` so that 
I can just use `print(f'whatever')` without needing to 
remember to use custom_logger.info.

In [2]:
now = datetime.datetime.now()
log_path = Path(GLOBAL_ECONOMY_LOG_DIR) / f"{now.year:04d}" / f"{now.month:02d}" / f"{now.day:02d}"
    
economy_logger = create_custom_logger(log_path, logger_name='economy')
original_print = print
override_print_with_logger(economy_logger)
economy_logger.info('Starting economy simulation at {}'.format(now.strftime("%Y-%m-%d %H:%M:%S")))

print_test_statements = True
if print_test_statements:
    # These are test lines to verify that the logger is working correctly. 
    # Set print_test_statements to False to disable them.
    economy_logger.debug("TEST: This is a DEBUG message (only in notebook)")
    economy_logger.info("TEST: This is an INFO message (in both file and notebook)")
    economy_logger.error("TEST: This is an ERROR message (in both file and notebook)")
    print("TEST: This is a test print statement (should appear in both file and notebook)")

2026-05-29 11:17:51,331 - economy - INFO - Starting economy simulation at 2026-05-29 11:17:51
2026-05-29 11:17:51,332 - economy - DEBUG - TEST: This is a DEBUG message (only in notebook)
2026-05-29 11:17:51,332 - economy - INFO - TEST: This is an INFO message (in both file and notebook)
2026-05-29 11:17:51,332 - economy - ERROR - TEST: This is an ERROR message (in both file and notebook)
2026-05-29 11:17:51,333 - economy - INFO - TEST: This is a test print statement (should appear in both file and notebook)


### Actor Objects

This cell contains objects for the different types of actors in the economy.

In [3]:
class BankAccount:
    balance: Decimal
    owner: str # uuid

    def __init__(self, initial_balance: Decimal = Decimal(0)):
        self.balance = initial_balance

class Bank:
    accounts: list[BankAccount]

    def __init__(self):
        self.accounts = []

    def create_account(self, owner: uuid) -> BankAccount:
        """
        Create a new bank account for the given owner UUID.

        Args:
            - owner (uuid): The UUID of the account owner.
        Returns:
            - BankAccount: The newly created bank account for the owner.
        """
        for account in self.accounts:
            if account.owner == owner:
                raise ValueError(f'Account already exists for owner {owner}')
        account = BankAccount()
        account.owner = owner
        self.accounts.append(account)
        return account
    
    
    def get_account_by_owner(self, owner: uuid) -> BankAccount:
        """
        Get a bank account by the owner's UUID.

        Args:
            - owner (uuid): The UUID of the account owner.
        Returns:
            - BankAccount: The bank account associated with the given owner UUID.
        Raises:
            - ValueError: If no account is found for the given owner UUID.
        """
        for account in self.accounts:
            if account.owner == owner:
                return account
        raise ValueError(f'No account found for owner {owner}')
    

    def transfer_money(self, sender: uuid, receiver: uuid, amount: Decimal):
        """
        Transfer money from one account to another.

        Args:
            - sender (uuid): The UUID of the sender's account.
            - receiver (uuid): The UUID of the receiver's account.
            - amount (Decimal): The amount of money to transfer.

        Returns:
            None

        Raises:
            - ValueError: If either the sender or receiver account is not found.
            - NotEnoughMoneyError: If the sender does not have enough money to transfer.
        """
        sender_account = self.get_account_by_owner(sender)
        receiver_account = self.get_account_by_owner(receiver)
        if sender_account.balance < amount:
            raise NotEnoughMoneyError(f'Sender {sender} does not have enough money to transfer {amount}. Current balance: {sender_account.balance}')
        sender_account.balance -= amount
        receiver_account.balance += amount




class Actor:
    name: str
    money: Decimal
    id: uuid

    def __init__(self, name: str, money: Decimal = Decimal(0)):
        self.name = name
        self.money = money
        self.id = uuid.uuid4()
        
    def reduce_money(self, amount: Decimal) -> Decimal:
        """
        Reduce an actor's money by a specified amount.

        Args:
            - amount (Decimal): The amount to reduce from the actor's money.
        Returns:
            - Decimal: The remaining balance in the actor's account after reduction.
        """
        if amount > self.money:
            raise NotEnoughMoneyError(f'{self.name} does not have enough money to reduce by {amount}. Current balance: {self.money}')
        self.money -= amount
        economy_logger.info(f'{self.name} reduced money by {amount}. New balance: {self.money}')
        return self.money


    def increase_money(self, amount: Decimal):
        """
        Increase an Actor's money by a specified amount.

        Args:
            - amount (Decimal): The amount to increase the actor's money by.
        Returns:
            - Decimal: The new balance in the actor's account after the increase.
        """
        self.money += amount
        economy_logger.info(f'{self.name} increased money by {amount}. New balance: {self.money}')
        return self.money
    

class Retailer(Actor):
    def __init__(self, name: str, retailer_config: RetailerConfig):
        super().__init__(name, money=retailer_config.starting_money)
        self.price = -1
        self.stock = 0
        self.competition_model = retailer_config.competition_model
        self.willing_to_sell = True if self.competition_model == CompetitionModel.BERTRAND else False
        if self.competition_model is CompetitionModel.COURNOT:
            raise NotImplementedError('Cournot competition model not implemented yet')

    def calculate_demand(self, wholesaler_stock: int, wholesaler_price: Decimal) -> int:
        """
        Calculates the demand for goods from this Retailer based on the wholesaler's
        stock and price.

        The retailer demands as many goods as they can afford at the wholesaler's px.

        Args:
            - wholesaler_stock (int): The current stock available at the wholesaler.
            - wholesaler_price (Decimal): The current price per unit at the wholesaler.
        Returns:
            - int: The quantity of goods the retailer demands from the wholesaler.
        """

        max_affordable_quantity = int(self.money // wholesaler_price)
        demanded_quantity = min(max_affordable_quantity, wholesaler_stock)

        print(f'{self.name} can afford up to {max_affordable_quantity} units at the wholesaler price of {wholesaler_price}. (has {self.money} money)')

        economy_logger.info(f'{self.name} is requesting {demanded_quantity} goods from the wholesaler. (Max affordable: {max_affordable_quantity}, Wholesaler stock: {wholesaler_stock})')
        return demanded_quantity
    

    def calculate_willing_to_sell(self, wholesaler_price: Decimal):
        """
        Determines if this Retailer is willing to sell goods to consumers based
        on the competition model, and the current wholesale price.

        Args:
            - wholesaler_price (Decimal): The current price per unit at the wholesaler.
        Returns:
            - bool: True if the retailer is willing to sell goods to consumers, False otherwise.
        """
        if self.competition_model == CompetitionModel.BERTRAND:
            # in Bertrand competition, retailers are always willing to sell to consumers
            # so long as they have stock, regardless of the wholesale price
            if self.stock <= 0:
                self.willing_to_sell = False
            else:
                self.willing_to_sell = True
        elif self.competition_model == CompetitionModel.COURNOT:
            raise NotImplementedError('Cournot competition model not implemented yet')
        else:
            raise ValueError(f'Unknown competition model: {self.competition_model}')
        return self.willing_to_sell

    
    def receive_goods(self, quantity):
        economy_logger.info(f'{self.name} received {quantity} units from the wholesaler...')
        self.stock += quantity
        return self.stock
    
    def set_price(self, price):
        economy_logger.info(f'{self.name} is setting price to {price}...')
        self.price = price

    def offer_goods(self):
        economy_logger.info(f'{self.name} is offering goods at price {self.price} each...')

    def process_sale(self):
        """
        Process the sale of a retailer's goods to a consumer. Reduce the retailer's
        stock by 1, and if the stock reaches 0, set the willing_to_sell flag to False.

        Returns:
            - None
        Raises:
            - ValueError: If the retailer cannot process the sale because stock is 0 or less
        """
        economy_logger.info(f'{self.name} is processing a sale...')
        if self.stock <= 0:
            raise ValueError(f'{self.name} cannot process sale because stock is {self.stock}')
        self.stock -= 1
        economy_logger.info(f'{self.name} has {self.stock} units left after the sale.')
        if self.stock <= 0:
            economy_logger.info(f'{self.name} has run out of stock and is no longer willing to sell to consumers.')
            self.willing_to_sell = False




class Wholesaler(Actor):
    price: Decimal
    stock: int
    is_unlimited_wholesaler: bool

    def __init__(self, name):
        super().__init__(name, money=Decimal(0))
        self.price = 0
        self.stock = 0
        self.is_unlimited_wholesaler = True

    def set_price(self, price: Decimal) -> None:
        """
        Set the price for the wholesaler's goods.

        Args:
            - price (Decimal): The price to set for the wholesaler's goods.
        Raises:
            - ValueError: If the price is not a Decimal.
        """
        if not isinstance(price, Decimal):
            raise ValueError(f'Price must be a Decimal, got {type(price)}')
        economy_logger.info(f'{self.name} is setting price to {price}...')
        self.price = price

    def set_stock(self, stock: int):
        """
        Set the stock for the wholesaler's goods.

        Args:
            - stock (int): The stock to set for the wholesaler's goods.
        Raises:
            - ValueError: If the stock is not an integer.
        """
        if not isinstance(stock, int):
            raise ValueError(f'Stock must be an integer, got {type(stock)}')
        economy_logger.info(f'{self.name} is setting stock to {stock}...')
        self.stock = stock


    def get_current_price(self):
        """
        Get the current price of the wholesaler's goods.
        """
        return self.price

    def offer_goods(self):
        economy_logger.info(f'{self.name} is selling up to {self.stock} units at price {self.price} each...')
        return self.stock, self.price

    def process_sale(self):
        """
        Process a sale of exactly one of the wholesaler's goods.
        """
        economy_logger.info(f'{self.name} is processing a sale...')
        if self.is_unlimited_wholesaler:
            economy_logger.info(f'{self.name} is an unlimited wholesaler, so stock remains unchanged.')
        else:
            self.stock -= 1


class Consumer(Actor):
    earns_per_iteration: Decimal
    basket_count: int # the number of goods the consumer is holding
    utility: Decimal # the consumer's utility, which increases with each good they consume
    
    def __init__(self, name, consumer_config: ConsumerConfig):
        super().__init__(name, money=Decimal(consumer_config.earns_per_iteration))
        self.earns_per_iteration = consumer_config.earns_per_iteration
        self.basket_count = 0
        self.utility = Decimal('0.0')

    def consume_basket(self):
        economy_logger.info(f'{self.name} is consuming their basket of goods...')
        self.utility += Decimal('10.0') * self.basket_count
        self.basket_count = 0


## The Economy

This cell contains the Economy class. That's the schema for the program, and contains the different types of Actors in the economy, and handles the way they interact with each other.

In [ ]:
class Economy:
    wholesaler: Wholesaler = None
    retailers: list[Retailer] = None
    consumers: list[Consumer] = None
    bank: Bank

    def __init__(self, 
                 wholesaler_config: WholesalerConfig,
                 retailer_config: RetailerConfig,
                 consumer_config: ConsumerConfig):
        self.wholesaler = self.setup_wholesaler(wholesaler_config)
        self.retailers = self.setup_retailers(retailer_config)
        self.consumers = self.setup_consumers(consumer_config)
        self.bank = Bank()


    def transfer_money(self, sender, receiver, amount):
        """
        Transfer money from one Actor to another, ensuring that the sender has sufficient funds and logging the transaction.
        Args:
            - sender (Actor): The Actor sending the money.
            - receiver (Actor): The Actor receiving the money.
            - amount (Decimal): The amount of money to transfer.
        Raises:
            - NotEnoughMoneyError: If the sender does not have enough money to complete the transfer.
        """
        print(f'{sender.name} is sending {amount} to {receiver.name}...')
        sender.reduce_money(amount)
        receiver.increase_money(amount)


    def process_wholesaler_transaction(self, wholesaler: Wholesaler,retailer: Retailer, quantity: int):
        """
        Process a transaction between a retailer and the wholesaler, including money transfer and inventory updates.

        Args:
            - wholesaler (Wholesaler): The wholesaler involved in the transaction.
            - retailer (Retailer): The retailer involved in the transaction.
            - quantity (int): The quantity of goods being purchased by the retailer from the wholesaler.
        """
        total_cost = wholesaler.get_current_price() * quantity
        print(f'{retailer.name} is buying {quantity} units from {wholesaler.name} for a total cost of {total_cost}...')
        try:
            self.transfer_money(retailer, wholesaler, total_cost)
            for _ in range(quantity):
                wholesaler.process_sale()
            retailer.receive_goods(quantity)
        except NotEnoughMoneyError as e:
            economy_logger.error(f'Transaction failed due to low funds: {e}')


    def process_retailer_transaction(self, retailer: Retailer, consumer: Consumer):
        """
        Process a transaction between a retailer and a consumer, including money transfer and inventory updates.

        Args:
            - retailer (Retailer): The retailer involved in the transaction.
            - consumer (Consumer): The consumer involved in the transaction.
        """
        total_cost = retailer.price
        print(f'{consumer.name} is buying 1 unit from {retailer.name} for a total cost of {total_cost}...')
        try:
            self.transfer_money(consumer, retailer, total_cost)
            retailer.process_sale()
            consumer.basket_count += 1
        except NotEnoughMoneyError as e:
            economy_logger.error(f'Transaction failed due to low funds: {e}')


    def iteration_report(self):
        print('--- Iteration Report ---')
        print(f'Wholesaler: {self.wholesaler.name}, Money: {self.wholesaler.money}, Stock: {self.wholesaler.stock}, Price: {self.wholesaler.price}')
        for retailer in self.retailers:
            print(f'Retailer: {retailer.name}, Money: {retailer.money}, Stock: {retailer.stock}, Price: {retailer.price}')
        for consumer in self.consumers:
            print(f'Consumer: {consumer.name}, Money: {consumer.money}, Utility: {consumer.utility}, Basket Count: {consumer.basket_count}')
        print('--- End of Report ---')


    def run_loop(self, current_iteration: int):
        economy_logger.info(f'Running economy loop for iteration {current_iteration}...')
        
        # wholesaler sets prices and offers goods

        print(f'--- Wholesaler Loop Iteration {current_iteration} ---')
        self.wholesaler.set_price(Decimal('10.0'))
        self.wholesaler.set_stock(1_000_000)
        wholesaler_stock, wholesaler_price = self.wholesaler.offer_goods()

        print(f'--- Retailer Loop Iteration {current_iteration} ---')

        for retailer in self.retailers:
            print(f'{retailer.name} has {retailer.stock} units in stock before the iteration')
            demanded_quantity = retailer.calculate_demand(wholesaler_stock, wholesaler_price)
            self.process_wholesaler_transaction(self.wholesaler, retailer, demanded_quantity)

            retailer.set_price(wholesaler_price * Decimal('1.5')) # TODO: impl a pricing strategy
            retailer.calculate_willing_to_sell(wholesaler_price)
            print(f'{retailer.name} has {retailer.stock} units in stock after the iteration setup, and is {"willing" if retailer.willing_to_sell else "not willing"} to sell.')



        print(f'--- Consumer Loop Iteration {current_iteration} ---')
        self.retailers.sort(key=lambda r: r.price)

        for consumer in self.consumers:
            print(f'{consumer.name} has {consumer.money} money, and {consumer.utility} utility before the iteration')
            
            print(f'{consumer.name} earns {consumer.earns_per_iteration} money at the start of the iteration...')
            consumer.money += consumer.earns_per_iteration # give the consumer their earnings for the iteration

            def find_retailer_to_buy_from(consumer, retailers):
                retailers_to_consider = [retailer for retailer in retailers if retailer.willing_to_sell]
                if not retailers_to_consider:
                    print(f'{consumer.name} has no retailers to consider for purchase. The consumer cannot purchase in this iteration.')
                    return None
                if consumer.money < min(retailer.price for retailer in retailers_to_consider):
                    print(f'{consumer.name} cannot afford any retailers. The consumer cannot purchase in this iteration.')
                    return None
                return retailers_to_consider[0] # return the cheapest retailer that is willing to sell
            
            while True:
                retailer_to_buy_from = find_retailer_to_buy_from(consumer, self.retailers)
                if retailer_to_buy_from is None:
                    break
                print(f'{consumer.name} is buying from {retailer_to_buy_from.name} at price {retailer_to_buy_from.price}...')
                try:
                    self.process_retailer_transaction(retailer_to_buy_from, consumer)
                except NotEnoughMoneyError as e:
                    economy_logger.error(f'Transaction between {consumer.name} and {retailer_to_buy_from.name} failed due to low funds: {e}')
                    break
            
            consumer.consume_basket() # consume the basket at the end of the iteration


        self.iteration_report()

        economy_logger.info(f'Economy loop for iteration {current_iteration} complete.\n\n')


    def setup_wholesaler(self, wholesaler_config: WholesalerConfig):
        return Wholesaler(wholesaler_config.name)

    def setup_consumers(self, consumer_config: ConsumerConfig):
        """
        Setup the consumers in the economy based on the provided configuration.

        Each consumer has an earning rate, defined in ConsumerConfig, which determines
        how much money they earn at the start of each iteration.

        Args:
            - consumer_config (ConsumerConfig): The configuration for setting up consumers
        Returns:
            - list[Consumer]: A list of Consumer instances set up according to the configuration.
        """
        def setup_consumer(name, consumer_config):
            return Consumer(name, consumer_config)
        
        consumers = []
        economy_logger.debug(f'Setting up {consumer_config.num_consumers} consumers...')
        for i in range(consumer_config.num_consumers):
            consumer = setup_consumer(f'Consumer {i+1}', consumer_config)
            consumers.append(consumer)
        return consumers


    def setup_retailers(self, retailer_config: RetailerConfig):
        def setup_retailer(name):
            return Retailer(name, retailer_config)
        
        retailers = []
        
        economy_logger.debug(f'Setting up {retailer_config.num_retailers} retailers...')

        for i in range(retailer_config.num_retailers):
            retailer = setup_retailer(f'Retailer {i+1}')
            retailers.append(retailer)
        return retailers

## Application Loop

This cell runs the application in a loop.

In [5]:
from IPython.display import clear_output
import ipywidgets as widgets


def main(iterations):
    wholesaler_config = WholesalerConfig()
    
    run_competition_model = CompetitionModel.BERTRAND
    retailer_config = RetailerConfig()
    retailer_config.competition_model = run_competition_model
    retailer_config.num_retailers = 3

    consumer_config = ConsumerConfig()
    consumer_config.num_consumers = 15
    consumer_config.earns_per_iteration = Decimal('150.0')
    
    economy = Economy(wholesaler_config, retailer_config, consumer_config)

    print(f'Starting {run_competition_model.name} economy simulation for {iterations} iterations...')
    for i in range(iterations):
        print(f'Running iteration {i + 1}...')
        economy.run_loop(i + 1)
        # clear_output(wait=True)
    print('Economy simulation complete after {} iterations.'.format(iterations))



main(2)

2026-05-29 11:17:51,399 - economy - DEBUG - Setting up 3 retailers...
2026-05-29 11:17:51,400 - economy - DEBUG - Setting up 15 consumers...
2026-05-29 11:17:51,400 - economy - INFO - Starting BERTRAND economy simulation for 2 iterations...
2026-05-29 11:17:51,401 - economy - INFO - Running iteration 1...
2026-05-29 11:17:51,401 - economy - INFO - Running economy loop for iteration 1...
2026-05-29 11:17:51,401 - economy - INFO - --- Wholesaler Loop Iteration 1 ---
2026-05-29 11:17:51,402 - economy - INFO - Orange Wholesaler is setting price to 10.0...
2026-05-29 11:17:51,402 - economy - INFO - Orange Wholesaler is setting stock to 1000000...
2026-05-29 11:17:51,402 - economy - INFO - Orange Wholesaler is selling up to 1000000 units at price 10.0 each...
2026-05-29 11:17:51,403 - economy - INFO - --- Retailer Loop Iteration 1 ---
2026-05-29 11:17:51,403 - economy - INFO - Retailer 1 has 0 units in stock before the iteration
2026-05-29 11:17:51,403 - economy - INFO - Retailer 1 can affor